In [22]:
import json
import re

In [23]:
def hms_to_seconds(hms):
    parts = [int(x) for x in hms.strip().split(':')]
    if len(parts) == 3:
        h, m, s = parts
    else:
        print("异常时间戳:", hms)
        raise ValueError(f"hms_to_seconds: 不支持的时间格式 '{hms}'")
    return h * 3600 + m * 60 + s

In [24]:
input_path = 'questions_sqa.json'  # 你的第一个JSON文件路径
output_path = 'question_sqa_rekv_online.json'  # 输出路径

In [21]:
# with open(input_path, "r", encoding="utf-8") as f:
#     data = json.load(f)

# video_groups = {}
# for idx, item in enumerate(data):
#     video_path = item['video_path']
#     video_id = video_path.split("/")[-1].replace('.mp4', '')
#     key = (video_id, video_path)
#     if key not in video_groups:
#         video_groups[key] = []
    
#     # 解析时间
#     time_match = re.match(r'\[(\d{1,2}:\d{2}:\d{2})\s*-\s*(\d{1,2}:\d{2}:\d{2})\]', item["time"])
#     if time_match:
#         start_str, _ = time_match.groups()
#         start_time = hms_to_seconds(start_str)
#     else:
#         start_time = 0
    
#     for q in item["questions"]:
#         # option去除编号
#         choices = []
#         for opt in q["options"]:
#             opt_clean = re.sub(r'^[A-Z]\.\s*', '', opt)  # 去"A. "前缀
#             choices.append(opt_clean)
        
#         answer_letter = q["answer"].strip().upper()
#         answer_idx = ord(answer_letter) - ord('A')
#         answer_content = choices[answer_idx] if 0 <= answer_idx < len(choices) else ""
        
#         # "end_time" 用 time_stamp 字段
#         end_time = hms_to_seconds(q["time_stamp"])
        
#         qa_item = {
#             "question": q["question"],
#             "choices": choices,
#             "answer": answer_content,
#             "question_type": q.get("task_type", ""),
#             "start_time": start_time,
#             "end_time": end_time
#         }
#         video_groups[key].append(qa_item)

# # 汇总输出
# output_data = []
# for (video_id, video_path), conversations in video_groups.items():
#     duration = max([qa["end_time"] for qa in conversations]) if conversations else 0
#     output_data.append({
#         "video_id": video_id,
#         "video_path": video_path,
#         "duration": duration,
#         "conversations": conversations
#     })

TypeError: list indices must be integers or slices, not str

In [25]:
with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)

video_groups = {}

for group in data:         # 外层list
    for item in group:     # 内层list，每个 item 是 dict
        video_path = item['video_path']
        video_id = video_path.split("/")[-1].replace('.mp4', '')
        key = (video_id, video_path)
        if key not in video_groups:
            video_groups[key] = []
        
        # 解析时间
        time_match = re.match(r'\[(\d{1,2}:\d{2}:\d{2})\s*-\s*(\d{1,2}:\d{2}:\d{2})\]', item["time"])
        if time_match:
            start_str, _ = time_match.groups()
            start_time = hms_to_seconds(start_str)
        else:
            start_time = 0
        
        for q in item["questions"]:
            # option去除编号
            choices = [re.sub(r'^[A-Z]\.\s*', '', opt) for opt in q["options"]]
            
            answer_letter = q["answer"].strip().upper()
            answer_idx = ord(answer_letter) - ord('A')
            answer_content = choices[answer_idx] if 0 <= answer_idx < len(choices) else ""
            
            # "end_time" 用 time_stamp 字段
            end_time = hms_to_seconds(q["time_stamp"])
            
            qa_item = {
                "question": q["question"],
                "choices": choices,
                "answer": answer_content,
                "question_type": q.get("task_type", ""),
                "start_time": start_time,
                "end_time": end_time
            }
            video_groups[key].append(qa_item)

# 汇总输出
output_data = []
for (video_id, video_path), conversations in video_groups.items():
    duration = max([qa["end_time"] for qa in conversations]) if conversations else 0
    output_data.append({
        "video_id": video_id,
        "video_path": video_path,
        "duration": duration,
        "conversations": conversations
    })

In [26]:
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"转换完成，见 {output_path}")

转换完成，见 question_sqa_rekv_online.json
